# Task 3 V2 — Static Malware Analysis

**Proof-of-concept:** Apply data analytics to Windows PE file analysis.

| Sub-task | Technique |
|----------|----------|
| 3.1 PE Headers | `pefile` → interactive Plotly table |
| 3.2 PE Sections | `pefile` → interactive Plotly table |
| 3.3 String Analysis | Printable string extraction + Shannon entropy + Plotly histogram |
| 3.4 Heuristic | Extended 10-API suspicious list · threshold 50% |
| 3.5 ML Classification | Random Forest + SVM (RBF) · ROC curves (Plotly) |

**Samples:** `File1.task3`, `File2.task3`, `File3.task3` (Windows PE binaries)  
**ML Dataset:** `data/dataset_task3.csv`

In [1]:
# Install required libraries
!pip install pefile plotly scikit-learn pandas numpy --quiet


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os
import math
import string
import warnings
import struct

import pefile
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, accuracy_score
)
from sklearn.pipeline import Pipeline
from IPython.display import display
warnings.filterwarnings('ignore')

# ── Path resolution ───────────────────────────────────────────────────────────
# __vsc_ipynb_file__ is set by the VS Code kernel to the notebook's absolute path.
# This ensures V2/data/ is found regardless of what the kernel's CWD is.
try:
    _NB_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _NB_DIR = os.getcwd()

# V2/data — preferred self-contained data directory
_V2_DATA = os.path.normpath(os.path.join(_NB_DIR, '..', 'data'))

def find_data_dir():
    '''Locate V2/data/ using the notebook file location, with CWD fallbacks.'''
    candidates = [
        _V2_DATA,                               # V2/data  (preferred — self-contained)
        os.path.join(_NB_DIR, 'data'),          # notebook-local data/
        'data', '../data', '../../data',        # CWD-relative fallbacks
    ]
    for path in candidates:
        full = os.path.normpath(os.path.abspath(path))
        if os.path.isfile(os.path.join(full, 'dataset_task3.csv')):
            return full
    raise FileNotFoundError('Cannot locate data/ directory containing dataset_task3.csv')

def find_pe_files():
    '''Locate PE sample files — checks V2/ then workspace root.'''
    # PE files may live in V2/ itself or in the workspace root
    search_dirs = [
        _NB_DIR,                                           # V2/Task3/
        os.path.normpath(os.path.join(_NB_DIR, '..')),    # V2/
        os.path.normpath(os.path.join(_NB_DIR, '../..')), # workspace root
        os.getcwd(),
    ]
    result = {}
    for name in ['File1', 'File2', 'File3']:
        fname = f'{name}.task3'
        for d in search_dirs:
            candidate = os.path.join(d, fname)
            if os.path.isfile(candidate):
                result[name] = candidate
                break
    return result

DATA_DIR = find_data_dir()
PE_FILES = find_pe_files()
DATASET_CSV = os.path.join(DATA_DIR, 'dataset_task3.csv')

print(f'Data directory : {DATA_DIR}')
for name, path in PE_FILES.items():
    size_kb = os.path.getsize(path) / 1024 if os.path.isfile(path) else 0
    print(f'  {name}: {path}  ({size_kb:.1f} KB)')

Root directory : /workspaces/2026-4-6-Data-Science-for-Cyber-Security
Data directory : /workspaces/2026-4-6-Data-Science-for-Cyber-Security/data
  File1: /workspaces/2026-4-6-Data-Science-for-Cyber-Security/File1.task3  (112.0 KB)
  File2: /workspaces/2026-4-6-Data-Science-for-Cyber-Security/File2.task3  (17.0 KB)
  File3: /workspaces/2026-4-6-Data-Science-for-Cyber-Security/File3.task3  (120.5 KB)


## Tasks 3.1 & 3.2 — PE Header and Section Analysis (Plotly Tables)
The `pefile` library parses Windows Portable Executable (PE) files.
Results are rendered as interactive `go.Table` figures instead of DataFrames.

In [3]:
# ── Helper: parse a PE file into header + section records ─────────────────────
def parse_pe(name: str, path: str) -> dict:
    '''
    Parse a PE file using the pefile library.

    Returns:
        dict with keys 'header' (flat dict of key attributes)
             and 'sections' (list of dicts, one per section).
    '''
    try:
        pe = pefile.PE(path)
    except Exception as e:
        print(f'  ERROR parsing {name}: {e}')
        return {'header': {}, 'sections': []}

    header = {
        'File': name,
        'Machine': hex(pe.FILE_HEADER.Machine),
        'TimeDateStamp': pe.FILE_HEADER.TimeDateStamp,
        'NumberOfSections': pe.FILE_HEADER.NumberOfSections,
        'Characteristics': hex(pe.FILE_HEADER.Characteristics),
        'SizeOfOptionalHeader': pe.FILE_HEADER.SizeOfOptionalHeader,
        'AddressOfEntryPoint': hex(pe.OPTIONAL_HEADER.AddressOfEntryPoint),
        'ImageBase': hex(pe.OPTIONAL_HEADER.ImageBase),
        'SizeOfImage': pe.OPTIONAL_HEADER.SizeOfImage,
        'Subsystem': pe.OPTIONAL_HEADER.Subsystem,
        'NumberOfRvaAndSizes': pe.OPTIONAL_HEADER.NumberOfRvaAndSizes,
        'FileSize_KB': round(os.path.getsize(path) / 1024, 2),
    }

    try:
        n_imports = sum(
            len(entry.imports)
            for entry in pe.DIRECTORY_ENTRY_IMPORT
        ) if hasattr(pe, 'DIRECTORY_ENTRY_IMPORT') else 0
        header['TotalImports'] = n_imports
    except Exception:
        header['TotalImports'] = 0

    sections = []
    for sec in pe.sections:
        name_str = sec.Name.decode(errors='replace').strip('\x00')
        data = sec.get_data()
        sections.append({
            'File': name,
            'Section': name_str,
            'VirtualAddress': hex(sec.VirtualAddress),
            'VirtualSize': sec.Misc_VirtualSize,
            'RawSize': sec.SizeOfRawData,
            'Characteristics': hex(sec.Characteristics),
            'Entropy': round(sec.get_entropy(), 4),
        })

    pe.close()
    return {'header': header, 'sections': sections}


# Parse all files
pe_data = {name: parse_pe(name, path) for name, path in PE_FILES.items()}

# ── Task 3.1: PE Headers as Plotly Table ─────────────────────────────────────
header_rows = [v['header'] for v in pe_data.values() if v['header']]
if header_rows:
    hdr_df = pd.DataFrame(header_rows).T.reset_index()
    hdr_df.columns = ['Attribute'] + [v['header']['File']
                                       for v in pe_data.values() if v['header']]
    fig_hdr = go.Figure(
        go.Table(
            header=dict(
                values=list(hdr_df.columns),
                fill_color='#2c3e50',
                font=dict(color='white', size=12),
                align='left'
            ),
            cells=dict(
                values=[hdr_df[col] for col in hdr_df.columns],
                fill_color=[['#1a1a2e', '#16213e'] * 20],
                font=dict(color='white', size=11),
                align='left'
            )
        )
    )
    fig_hdr.update_layout(
        title='Task 3.1 — PE Header Attributes (Interactive Table)',
        template='plotly_dark',
        height=550
    )
    fig_hdr.show()

In [4]:
# ── Task 3.2: PE Sections as Plotly Table ─────────────────────────────────────
all_sections = []
for v in pe_data.values():
    all_sections.extend(v['sections'])

if all_sections:
    sec_df = pd.DataFrame(all_sections)

    fig_sec = go.Figure(
        go.Table(
            header=dict(
                values=list(sec_df.columns),
                fill_color='#2c3e50',
                font=dict(color='white', size=12),
                align='left'
            ),
            cells=dict(
                values=[sec_df[col] for col in sec_df.columns],
                fill_color=[
                    ['#1a252f' if row % 2 == 0 else '#16213e'
                     for row in range(len(sec_df))]
                ],
                font=dict(color='white', size=11),
                align='left'
            )
        )
    )
    fig_sec.update_layout(
        title='Task 3.2 — PE Section Details (Interactive Table)',
        template='plotly_dark',
        height=450
    )
    fig_sec.show()

    # Section entropy bar chart
    fig_ent = px.bar(
        sec_df, x='Section', y='Entropy', color='File',
        barmode='group',
        title='Task 3.2 — Section Entropy per File (>7 suggests packing/encryption)',
        labels={'Entropy': 'Shannon Entropy', 'Section': 'PE Section'},
        template='plotly_dark',
        text='Entropy'
    )
    fig_ent.add_hline(
        y=7.0, line_dash='dash', line_color='red',
        annotation_text='High entropy threshold (7.0)'
    )
    fig_ent.update_traces(texttemplate='%{text:.2f}', textposition='outside')
    fig_ent.show()

## Task 3.3 — String Extraction + Shannon Entropy Analysis
Printable ASCII strings (length ≥ 4) are extracted from raw bytes.
Shannon entropy is computed per file — high entropy (> 7.0) indicates
packed or encrypted content. String length distributions are plotted
as Plotly histograms.

In [5]:
# ── Task 3.3: String Extraction + Shannon Entropy ────────────────────────────

def extract_strings(file_bytes: bytes, min_len: int = 4) -> list:
    '''
    Extract sequences of printable ASCII characters from a byte sequence.

    Parameters:
        file_bytes (bytes): Raw binary content of the file.
        min_len (int): Minimum string length to keep.

    Returns:
        list of str: Extracted printable strings.
    '''
    printable = set(string.printable)
    result, current = [], []
    for byte in file_bytes:
        ch = chr(byte)
        if ch in printable and ch not in ('\n', '\r', '\t'):
            current.append(ch)
        else:
            if len(current) >= min_len:
                result.append(''.join(current))
            current = []
    if len(current) >= min_len:
        result.append(''.join(current))
    return result


def shannon_entropy(data: bytes) -> float:
    '''
    Compute Shannon entropy (bits) of a byte sequence.

    A value > 7.0 typically indicates packing or encryption.

    Parameters:
        data (bytes): Raw binary content.

    Returns:
        float: Entropy value in bits (0–8 range).
    '''
    if not data:
        return 0.0
    freq = {}
    for b in data:
        freq[b] = freq.get(b, 0) + 1
    n = len(data)
    return -sum((c / n) * math.log2(c / n) for c in freq.values())


string_data = []
entropy_summary = []

for name, path in PE_FILES.items():
    with open(path, 'rb') as f:
        raw = f.read()

    file_entropy = shannon_entropy(raw)
    strings = extract_strings(raw)
    lengths = [len(s) for s in strings]

    entropy_summary.append({
        'File': name,
        'Whole-file Entropy': round(file_entropy, 4),
        'Packed/Encrypted?': 'Likely' if file_entropy > 7.0 else 'Unlikely',
        'Total Strings': len(strings),
        'Avg String Length': round(np.mean(lengths), 2) if lengths else 0,
    })

    for s in strings:
        string_data.append({'File': name, 'String': s, 'Length': len(s)})

    print(f'{name}: entropy={file_entropy:.4f}, strings={len(strings)}')

# Entropy summary table
ent_df = pd.DataFrame(entropy_summary)
display(ent_df)

# Entropy bar chart
fig_ent_sum = px.bar(
    ent_df, x='File', y='Whole-file Entropy',
    title='Task 3.3 — Shannon Entropy per File (threshold 7.0)',
    color='Whole-file Entropy',
    color_continuous_scale=['#2ecc71', '#f39c12', '#e74c3c'],
    range_color=[5, 8],
    template='plotly_dark',
    text='Whole-file Entropy'
)
fig_ent_sum.add_hline(
    y=7.0, line_dash='dash', line_color='red',
    annotation_text='Packing threshold (7.0)'
)
fig_ent_sum.update_traces(texttemplate='%{text:.4f}', textposition='outside')
fig_ent_sum.update_layout(height=420, xaxis_title='File', yaxis_title='Shannon Entropy (bits)')
fig_ent_sum.show()

# String length distribution histograms
str_df = pd.DataFrame(string_data)
if not str_df.empty:
    fig_hist = px.histogram(
        str_df, x='Length', color='File',
        barmode='overlay',
        nbins=40,
        title='Task 3.3 — String Length Distribution per File',
        labels={'Length': 'String Length (chars)', 'count': 'Frequency'},
        opacity=0.7,
        template='plotly_dark'
    )
    fig_hist.update_layout(
        xaxis_title='String Length (characters)',
        yaxis_title='Number of Strings',
        legend_title='File',
        height=430
    )
    fig_hist.show()

File1: entropy=6.0036, strings=459
File2: entropy=5.3313, strings=129
File3: entropy=5.9561, strings=812


,File,Whole-file Entropy,Packed/Encrypted?,Total Strings,Avg String Length
0,File1,6.0036,Unlikely,459,8.89
1,File2,5.3313,Unlikely,129,8.64
2,File3,5.9561,Unlikely,812,8.37


## Task 3.4 — Extended Heuristic API Analysis (10 APIs)
Ten known suspicious Windows API calls are checked against each PE's
import table. If ≥ 50% of the suspicious APIs are present, the file
is flagged as **SUSPICIOUS**.

In [6]:
# ── Task 3.4: Heuristic API Analysis (10 suspicious APIs) ────────────────────

# Extended list of 10 suspicious Windows API calls
SUSPICIOUS_APIS = [
    'VirtualAlloc',
    'WriteProcessMemory',
    'CreateRemoteThread',
    'WinExec',
    'ShellExecuteA',
    'LoadLibraryA',      # dynamically loads DLLs at runtime
    'OpenProcess',       # accesses other processes (injection)
    'RegSetValueEx',     # modifies registry (persistence)
    'InternetOpenUrl',   # network communication
    'CryptEncrypt',      # encrypts data (ransomware indicator)
]

THRESHOLD_PCT = 0.50   # 50% threshold

heuristic_results = []

for name, path in PE_FILES.items():
    try:
        pe = pefile.PE(path)
        imported_apis = set()
        if hasattr(pe, 'DIRECTORY_ENTRY_IMPORT'):
            for entry in pe.DIRECTORY_ENTRY_IMPORT:
                for imp in entry.imports:
                    if imp.name:
                        imported_apis.add(imp.name.decode(errors='replace'))
        pe.close()
    except Exception as e:
        print(f'  ERROR reading imports from {name}: {e}')
        imported_apis = set()

    matches = [api for api in SUSPICIOUS_APIS if api in imported_apis]
    match_pct = len(matches) / len(SUSPICIOUS_APIS)
    verdict = 'SUSPICIOUS' if match_pct >= THRESHOLD_PCT else 'BENIGN'

    heuristic_results.append({
        'File': name,
        'Total Imports': len(imported_apis),
        'Suspicious APIs Found': len(matches),
        'Match %': round(match_pct * 100, 1),
        'Matched APIs': ', '.join(matches) if matches else 'None',
        'Verdict': verdict,
    })
    print(f'{name}: {len(matches)}/{len(SUSPICIOUS_APIS)} APIs matched → {verdict}')

heur_df = pd.DataFrame(heuristic_results)
display(heur_df)

# ── Visualise as Plotly bar chart ─────────────────────────────────────────────
colour_map = {'SUSPICIOUS': '#e74c3c', 'BENIGN': '#2ecc71'}
heur_df['Colour'] = heur_df['Verdict'].map(colour_map)

fig_heur = go.Figure()
for _, row in heur_df.iterrows():
    fig_heur.add_trace(
        go.Bar(
            x=[row['File']],
            y=[row['Match %']],
            name=row['File'],
            marker_color=row['Colour'],
            text=[f"{row['Verdict']}"],
            textposition='outside',
            textfont=dict(color=row['Colour'], size=13)
        )
    )

fig_heur.add_hline(
    y=50, line_dash='dash', line_color='red',
    annotation_text='Threshold (50%)', annotation_font_color='red'
)
fig_heur.update_layout(
    title='Task 3.4 — Suspicious API Match % per PE File (10-API Heuristic)',
    xaxis_title='File',
    yaxis_title='Suspicious API Match (%)',
    yaxis=dict(tickformat='.0f', ticksuffix='%', range=[0, 115]),
    template='plotly_dark',
    showlegend=False,
    height=450
)
fig_heur.show()

File1: 1/10 APIs matched → BENIGN
File2: 0/10 APIs matched → BENIGN
File3: 1/10 APIs matched → BENIGN


,File,Total Imports,Suspicious APIs Found,Match %,Matched APIs,Verdict
0,File1,132,1,10.0,LoadLibraryA,BENIGN
1,File2,57,0,0.0,None,BENIGN
2,File3,284,1,10.0,LoadLibraryA,BENIGN


## Task 3.5 — ML Classification: Random Forest vs SVM with ROC Curves
Two classifiers are trained on `dataset_task3.csv` (PE-derived features):
- **Random Forest** (100 trees)
- **SVM** with RBF kernel (probability estimates enabled)

Performance is compared with a Plotly ROC curve chart.

In [7]:
# ── Task 3.5: Random Forest + SVM with ROC Curves ────────────────────────────

# Load dataset
ml_df = pd.read_csv(DATASET_CSV)
print(f'Dataset shape: {ml_df.shape}')
print(f'Label distribution:\n{ml_df["label"].value_counts()}')
display(ml_df.head(3))

# Features and target
feature_cols = [c for c in ml_df.columns if c != 'label']
X = ml_df[feature_cols].values
y = ml_df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# ── Model 1: Random Forest ────────────────────────────────────────────────────
rf_model = Pipeline([
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

print('\n--- Random Forest ---')
print(classification_report(y_test, rf_pred,
                             target_names=['Benign', 'Malicious']))

# ── Model 2: SVM with RBF kernel ──────────────────────────────────────────────
svm_model = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='rbf', probability=True, random_state=42))
])
svm_model.fit(X_train, y_train)
svm_pred = svm_model.predict(X_test)
svm_proba = svm_model.predict_proba(X_test)[:, 1]

print('\n--- SVM (RBF kernel) ---')
print(classification_report(y_test, svm_pred,
                             target_names=['Benign', 'Malicious']))

# ── Confusion matrices (Plotly) ───────────────────────────────────────────────
for model_name, pred in [('Random Forest', rf_pred), ('SVM (RBF)', svm_pred)]:
    cm = confusion_matrix(y_test, pred)
    labels = ['Benign', 'Malicious']
    fig_cm = go.Figure(
        go.Heatmap(
            z=cm, x=labels, y=labels,
            colorscale='Blues',
            text=cm, texttemplate='%{text}',
            hovertemplate='True: %{y}<br>Pred: %{x}<br>Count: %{z}<extra></extra>'
        )
    )
    fig_cm.update_layout(
        title=f'Task 3.5 — Confusion Matrix: {model_name}',
        xaxis_title='Predicted Label',
        yaxis_title='True Label',
        template='plotly_dark',
        height=420
    )
    fig_cm.show()

# ── ROC Curves on the same Plotly chart ──────────────────────────────────────
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_proba)
fpr_svm, tpr_svm, _ = roc_curve(y_test, svm_proba)
auc_rf = auc(fpr_rf, tpr_rf)
auc_svm = auc(fpr_svm, tpr_svm)

fig_roc = go.Figure()
fig_roc.add_trace(
    go.Scatter(
        x=fpr_rf, y=tpr_rf,
        mode='lines',
        name=f'Random Forest (AUC = {auc_rf:.3f})',
        line=dict(color='#3498db', width=2.5)
    )
)
fig_roc.add_trace(
    go.Scatter(
        x=fpr_svm, y=tpr_svm,
        mode='lines',
        name=f'SVM RBF (AUC = {auc_svm:.3f})',
        line=dict(color='#e74c3c', width=2.5)
    )
)
# Diagonal (random classifier)
fig_roc.add_trace(
    go.Scatter(
        x=[0, 1], y=[0, 1],
        mode='lines',
        name='Random Classifier (AUC = 0.500)',
        line=dict(color='grey', dash='dash', width=1.5)
    )
)
fig_roc.update_layout(
    title='Task 3.5 — ROC Curves: Random Forest vs SVM (RBF Kernel)',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    xaxis=dict(range=[0, 1]),
    yaxis=dict(range=[0, 1.02]),
    template='plotly_dark',
    legend=dict(x=0.6, y=0.1),
    height=500
)
fig_roc.show()

# ── Model comparison table ────────────────────────────────────────────────────
from sklearn.metrics import precision_score, recall_score, f1_score
comparison = pd.DataFrame([
    {
        'Model': 'Random Forest',
        'Accuracy': round(accuracy_score(y_test, rf_pred), 4),
        'Precision': round(precision_score(y_test, rf_pred, average='weighted'), 4),
        'Recall': round(recall_score(y_test, rf_pred, average='weighted'), 4),
        'F1-Score': round(f1_score(y_test, rf_pred, average='weighted'), 4),
        'AUC': round(auc_rf, 4),
    },
    {
        'Model': 'SVM (RBF)',
        'Accuracy': round(accuracy_score(y_test, svm_pred), 4),
        'Precision': round(precision_score(y_test, svm_pred, average='weighted'), 4),
        'Recall': round(recall_score(y_test, svm_pred, average='weighted'), 4),
        'F1-Score': round(f1_score(y_test, svm_pred, average='weighted'), 4),
        'AUC': round(auc_svm, 4),
    }
])
print('\nModel Comparison:')
display(comparison)

# Plotly grouped bar: metric comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
fig_comp = go.Figure()
colours = ['#3498db', '#e74c3c']
for i, row in comparison.iterrows():
    fig_comp.add_trace(
        go.Bar(
            name=row['Model'],
            x=metrics,
            y=[row[m] for m in metrics],
            marker_color=colours[i],
            text=[f"{row[m]:.3f}" for m in metrics],
            textposition='outside'
        )
    )
fig_comp.update_layout(
    title='Task 3.5 — Model Comparison: Random Forest vs SVM (RBF)',
    xaxis_title='Metric',
    yaxis_title='Score',
    yaxis=dict(range=[0, 1.15]),
    barmode='group',
    template='plotly_dark',
    legend_title='Model',
    height=450
)
fig_comp.show()

Dataset shape: (100, 11)
Label distribution:
label
0    50
1    50
Name: count, dtype: int64


,size_kb,num_imports,num_sections,entropy_avg,num_strings_url,api_network,api_process,max_section_entropy,num_resources,size_code,label
0,450,112,5,5.21,2,0,4,6.31,12,32000,0
1,120,45,3,7.82,15,8,12,7.91,2,15000,1
2,890,250,6,4.84,0,0,2,5.12,45,120000,0



--- Random Forest ---
              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00        15
   Malicious       1.00      1.00      1.00        15

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30


--- SVM (RBF kernel) ---
              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00        15
   Malicious       1.00      1.00      1.00        15

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30




Model Comparison:


,Model,Accuracy,Precision,Recall,F1-Score,AUC
0,Random Forest,1.0,1.0,1.0,1.0,1.0
1,SVM (RBF),1.0,1.0,1.0,1.0,1.0


## Summary

| Sub-task | Technique | Output |
|----------|-----------|--------|
| 3.1 PE Headers | `pefile` + `go.Table` | Interactive header table |
| 3.2 PE Sections | `pefile` + `go.Table` | Sections table + entropy chart |
| 3.3 Strings + Entropy | Custom extractor + Shannon entropy | Entropy bars + length histogram |
| 3.4 Heuristic | 10 suspicious APIs, 50% threshold | Verdict per file |
| 3.5 ML | Random Forest + SVM (RBF) | ROC curves + metric comparison |